Connect to hosted runtime with T4

Git Clone to get Repo

In [3]:
!git clone https://github.com/ChristianDe-fabolous/3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics.git

Cloning into '3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics'...
remote: Enumerating objects: 14996, done.
remote: Counting objects: 100% (1081/1081), done.
remote: Compressing objects: 100% (1062/1062), done.
remote: Total 14996 (delta 19), reused 1061 (delta 19), pack-reused 13915 (from 4)
Receiving objects: 100% (14996/14996), 600.75 MiB | 40.95 MiB/s, done.
Resolving deltas: 100% (1430/1430), done.
Updating files: 100% (19019/19019), done.


In [4]:
!git pull

fatal: not a git repository (or any of the parent directories): .git


Enter repo

In [5]:
%cd 3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics

/content/3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics


Install requirements

In [6]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 57.6 MB/s eta 0:00:00:00:0100:01


Mount Google Drive for persistent output storage

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Configure output and log directories on Drive

In [9]:
import os
DRIVE_OUTPUT = "/content/drive/MyDrive/vlm_benchmark/outputs"
DRIVE_LOGS   = "/content/drive/MyDrive/vlm_benchmark/logs"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(DRIVE_LOGS, exist_ok=True)
os.environ["VLM_OUTPUT_DIR"] = DRIVE_OUTPUT
os.environ["VLM_LOG_DIR"]    = DRIVE_LOGS
print(f"Outputs -> {DRIVE_OUTPUT}")
print(f"Logs    -> {DRIVE_LOGS}")

Outputs -> /content/drive/MyDrive/vlm_benchmark/outputs
Logs    -> /content/drive/MyDrive/vlm_benchmark/logs


## Action Phase Dataset

Run inference on the full action-phase benchmark.
Results are saved per-run to Google Drive with full metadata for analysis.

In [ ]:
# Model to evaluate — change here only
AP_MODEL  = "qwen-7b"   # qwen-3b | qwen-7b | qwen-7b-int8 | qwen3-2b | qwen-32b-int8
AP_RUN_ID = f"action_phase_{AP_MODEL}_v1"
print(f"Run ID: {AP_RUN_ID}")

### Run all question types at once

In [ ]:
!python src/main.py \
    --task action_phase \
    --model {AP_MODEL} \
    --action-phase-data data/action_phase_dataset.jsonl \
    --image-root /content/3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics \
    --run-id {AP_RUN_ID} \
    --resume

### Run a single question type (optional)

In [ ]:
# choices: action_phase_id | progress | next_action | phase_success | task_success
AP_TYPE = "action_phase_id"
!python src/main.py \
    --task action_phase \
    --model {AP_MODEL} \
    --action-phase-data data/action_phase_dataset.jsonl \
    --image-root /content/3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics \
    --action-phase-type {AP_TYPE} \
    --run-id {AP_RUN_ID}_{AP_TYPE} \
    --resume

In [ ]:
import json, os

smoke_path = f"{DRIVE_OUTPUT}/{SMOKE_RUN_ID}/results.jsonl"
if not os.path.exists(smoke_path):
    print("No results yet — run the cell above first.")
else:
    smoke = [json.loads(l) for l in open(smoke_path) if l.strip()]
    print(f"{len(smoke)} results\n{'='*80}")
    for r in smoke:
        correct_mark = "✓" if r["correct"] else "✗"
        print(f"\n[{correct_mark}] id={r['entry_id']}  type={r.get('question_type','?')}  variant={r.get('variant','?')}")
        print(f"    GT: {r['ground_truth_label']}  |  Predicted: {r['predicted_label']}")
        print(f"    Description + answer:\n      {r['response_raw']}")
        print("-"*80)

In [ ]:
!python src/main.py \
    --task action_phase \
    --model {AP_MODEL} \
    --action-phase-data data/action_phase_dataset.jsonl \
    --image-root /content/3D-vision-Benchmarking-Spatial-and-State-Reasoning-Skills-of-VLMs-for-Robotics \
    --limit {SMOKE_N} \
    --describe \
    --run-id {SMOKE_RUN_ID} \
    --resume

In [ ]:
SMOKE_N        = 10   # number of questions to run
SMOKE_RUN_ID   = f"{AP_RUN_ID}_smoke"
print(f"Smoke run ID : {SMOKE_RUN_ID}")
print(f"Questions    : first {SMOKE_N} from data/action_phase_dataset.jsonl (combined view)")

### Smoke test — first N questions with visual description

Runs the first `SMOKE_N` entries from the combined action-phase dataset.
The model is asked to **describe what it sees** before giving its answer letter.
Useful for sanity-checking image loading, prompt formatting, and model output quality.
Results are saved to `{DRIVE_OUTPUT}/{AP_RUN_ID}_smoke/` and resume automatically on restart.

### Inspect results

Results are written to `{DRIVE_OUTPUT}/{AP_RUN_ID}/results.jsonl`.
Each line is one answered question with full metadata.

In [ ]:
import json, os
results_path = f"{DRIVE_OUTPUT}/{AP_RUN_ID}/results.jsonl"
if os.path.exists(results_path):
    results = [json.loads(l) for l in open(results_path) if l.strip()]
    print(f"Total answered: {len(results)}")
    from collections import Counter
    by_type = Counter(r['question_type'] for r in results)
    by_variant = Counter((r['question_type'], r.get('variant')) for r in results)
    correct_by_type = {}
    for qt in by_type:
        sub = [r for r in results if r['question_type'] == qt]
        correct_by_type[qt] = sum(r['correct'] for r in sub) / len(sub)
    print('\nAccuracy by question type:')
    for qt, acc in sorted(correct_by_type.items()):
        print(f'  {qt}: {acc:.1%} ({by_type[qt]} entries)')
    print('\nCounts by (type, variant):')
    for (qt, v), cnt in sorted(by_variant.items()):
        sub = [r for r in results if r['question_type']==qt and r.get('variant')==v]
        acc = sum(r['correct'] for r in sub) / len(sub) if sub else 0
        print(f'  {qt}/{v}: {acc:.1%} ({cnt})')
else:
    print('No results yet — run inference first.')

### Context vs no-context comparison (variant A vs B)

In [ ]:
if os.path.exists(results_path):
    print('Accuracy A (no context) vs B (with action phase sequence):\n')
    for qt in ['action_phase_id', 'progress', 'next_action', 'phase_success', 'task_success']:
        for v in ['A', 'B']:
            sub = [r for r in results if r['question_type']==qt and r.get('variant')==v]
            if sub:
                acc = sum(r['correct'] for r in sub) / len(sub)
                print(f'  {qt}/{v}: {acc:.1%} (n={len(sub)})')
        print()

### Answer distribution analysis

In [ ]:
if os.path.exists(results_path):
    print('Answer distribution (what the model actually picked):\n')
    for qt in sorted(by_type):
        sub = [r for r in results if r['question_type']==qt]
        dist = Counter(r.get('predicted_label') for r in sub)
        gt_dist = Counter(r.get('ground_truth_label') for r in sub)
        print(f'{qt}:')
        print(f'  GT  distribution: {dict(gt_dist)}')
        print(f'  Pred distribution: {dict(dist)}')
        print()